## Read the md5 checksums

In [ ]:
import requests
import os
from tqdm import tqdm

natroot = "https://astroarchive.noirlab.edu"
adsurl = f"{natroot}/api/adv_search"
DATA_DIR = "/global/homes/s/seanmacb/decam_template_tools/endurance/data"

with open(os.path.join(DATA_DIR, "pilot_elais_s1.txt"), "r") as f:
    exps = [int(line.strip()) for line in f]

# 1. Sort the list and create a fast-lookup set
exps = sorted(exps)
target_exps = set(exps)

# 2. Group the numbers into tightly packed ranges (spans of 500)
chunks = []
current_chunk = [exps[0]]

for e in exps[1:]:
    # If the gap between the lowest number in the chunk and the current number 
    # is greater than 500, cap the chunk and start a new one. 
    if e - current_chunk[0] > 500:
        chunks.append(current_chunk)
        current_chunk = [e]
    else:
        current_chunk.append(e)
chunks.append(current_chunk)

md5s = {}

# 3. Query the API using min/max bounds for each chunk
for chunk in tqdm(chunks, desc="Querying chunks"):
    min_e = chunk[0]
    max_e = chunk[-1]
    
    jj = {
        "outfields": [
            "md5sum",
            "archive_filename",
            "instrument",
            "proc_type",
            "EXPNUM", 
        ],
        "search": [
            ["instrument", "decam"],
            ["proc_type", "raw"],
            ["EXPNUM", min_e, max_e],  # The API natively understands this as a range
        ]
    }
    
    # Request up to 1000 items (the max for NOIRLab) to ensure we get the whole range block
    response = requests.post(f'{adsurl}/find/?limit=1000', json=jj)
    
    if response.status_code != 200:
        print(f"\nHTTP Error {response.status_code}: {response.text}")
        continue
        
    res = response.json()
    
    if isinstance(res, dict):
        print(f"\nAPI Error Message: {res}")
        continue
        
    records = res[1:]
    
    # 4. Filter the returned block in Python
    for rec in records:
        if 'EXPNUM' in rec:
            rec_exp = int(rec['EXPNUM'])
            # Only save the md5sum if it's one of the exact exposures we wanted!
            if rec_exp in target_exps:
                md5s[rec_exp] = rec['md5sum']

print(f"Successfully retrieved {len(md5s)} md5sums!")

## Download the images from the checksums

In [1]:
with open("../data/pilot_bias_md5s.txt", "r") as file:
    md5s = file.read().splitlines()

print(f"Successfully retrieved {len(md5s)} md5sums!")

Successfully retrieved 833 md5sums!


In [2]:
import os
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# 1. Define your target directory
OUTPUT_DIR = "/pscratch/sd/e/elhoward/ELAIS_S1_pilot/calib/bias"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Setup a Session to reuse the underlying TCP connection
session = requests.Session()

# Optional: If these exposures are proprietary, uncomment the line below and add your token
# session.headers.update({"Authorization": "Token YOUR_API_TOKEN_HERE"})

def download_fits(md5):
    """
    Downloads a single FITS file in chunks to minimize RAM usage.
    """
    url = f"https://astroarchive.noirlab.edu/api/retrieve/{md5}/"
    
    # DECam raw files are almost always fpack compressed, so .fits.fz is appropriate
    output_path = os.path.join(OUTPUT_DIR, f"DECam_bias_{md5}.fits.fz")
    
    # Fault-tolerance: Skip if we already downloaded this file successfully
    if os.path.exists(output_path):
        return True

    try:
        with session.get(url, stream=True) as r:
            r.raise_for_status()

            with open(output_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

        return True
    except Exception as e:
        print(f"\nError downloading {md5}: {e}")
        return False

# 3. Download in parallel (5 workers is a safe sweet spot for NOIRLab limits)
max_workers = 6

print(f"Starting download of {len(md5s)} FITS files...")

# 4. Use ThreadPoolExecutor to download multiple files simultaneously
with tqdm(total=len(md5s), desc="Downloading Exposures") as pbar:
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        
        # Submit all tasks to the thread pool
        futures = {executor.submit(download_fits, md5): md5 for md5 in md5s}
        
        # Update progress bar as each thread completes its download
        for future in as_completed(futures):
            pbar.update(1)

print("All downloads complete!")

Starting download of 833 FITS files...


All downloads complete!
